<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_27_practicum_dp_greedy/note_lesson_27_scheduling_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚕 Урок 27 — Практикум П5: розклад водія (greedy + динамічне програмування)

Водій «Смачно + Таксі» обирає замовлення на зміну. Дві мети на тих самих даних:

1. **найбільше замовлень** — жадібний алгоритм за найранішим кінцем;
2. **найбільший заробіток** — динамічне програмування «взяти або пропустити».

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| Р | розминка: сходи — перекриття підзадач, словник-пам'ять, `@cache`, таблиця | Р.1–Р.4 | |
| 0 | дані, модель, текстова часова схема | 0.1 | |
| 1 | сумісність двох замовлень, таблиця сумісності | 1.1–1.2 | |
| 2 | мета 1: найбільше замовлень (greedy), покроковий журнал, правила, що ламаються | 2.1–2.3 | |
| 3 | зміна мети: гроші; «найдорожче» і «найвигідніше за годину» теж ламаються | 3.1–3.3 | **Задача 1** |
| 4 | `p(i)` — останнє сумісне замовлення | 4.1–4.2 | |
| 5 | таблиця ДП «взяти або пропустити» покроково, помилка на межі | 5.1–5.4 | |
| 6 | який розклад дав максимум: відновлення з кінця | 6.1–6.2 | **Задача 2** |
| 7 | те саме зверху вниз: рекурсія, `lru_cache`, чотири питання ДП | 7.1–7.3 | |
| 8 | перевірка перебором: граничні й випадкові набори, уся зміна | 8.1–8.4 | |
| 9 | складність `O(n²)` і `bisect` | 9.1–9.2 | **Задача 3** |
| 10 | решта монетами: greedy, контрприклад, таблиця, вибір монет, матрична параметризація | 10.1–10.6 | **Задача 4** |
| 11 | як читати помилки теми | 11.1–11.3 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**: приклади спираються на функції з попередніх.
- Клітинка з визначенням (`def`, клас, дані) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Теорія, аргумент обміну й складність — у книзі: [Урок 27](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_27/).

---
## Р. Розминка: сходи

Скількома способами піднятися на `n` сходинок, якщо за крок долаєш 1 або 2? Останній крок був або на 1, або на 2 сходинки, тож

`stairs(n) = stairs(n - 1) + stairs(n - 2)`, `stairs(0) = 1` (нуль сходинок — один спосіб: ти вже нагорі), `stairs(1) = 1`.

### Приклад Р.1. Звичайна рекурсія і лічильник викликів

In [ ]:
calls = 0


def stairs(n):
    global calls
    calls += 1
    if n <= 1:
        return 1
    return stairs(n - 1) + stairs(n - 2)

In [ ]:
for n in (10, 20, 30):
    calls = 0
    print(f"stairs({n}) = {stairs(n)}, викликів {calls}")

Для 30 сходинок — 2 692 537 викликів: `stairs(28)` рахується двічі, `stairs(27)` — тричі, `stairs(26)` — п'ять разів… Ті самі менші питання всередині більшого — це **перекриття підзадач** (overlapping subproblems). Знак, що відповіді варто запам'ятати.

### Приклад Р.2. Мемоізація словником

Перед обчисленням перевіряємо, чи `n` уже в словнику; після обчислення — записуємо. Лічильник рахує **усі** виклики, і ті, що лише підглянули.

In [ ]:
memo = {}


def stairs_memo(n):
    global calls
    calls += 1
    if n in memo:
        if n >= 27:
            print(f"  stairs_memo({n}) — уже в словнику")
        return memo[n]
    if n <= 1:
        result = 1
    else:
        result = stairs_memo(n - 1) + stairs_memo(n - 2)
    memo[n] = result
    return result

In [ ]:
calls = 0
memo.clear()
print(stairs_memo(30))
print("викликів:", calls, "| у словнику:", len(memo))

59 викликів замість 2 692 537: 31 справжнє обчислення (`n` від 0 до 30) і 28 підглядань. Вище видно останні два: `stairs_memo(29)` удруге питає `stairs_memo(27)`, а `stairs_memo(30)` — `stairs_memo(28)`, і обидва вже в словнику.

### Приклад Р.3. `@functools.cache` — те саме готовим декоратором (урок 19)

In [ ]:
from functools import cache


@cache
def stairs_cached(n):
    if n <= 1:
        return 1
    return stairs_cached(n - 1) + stairs_cached(n - 2)

In [ ]:
print(stairs_cached(30))
print(stairs_cached.cache_info())

`misses=31` — справжні обчислення, `hits=28` — відповіді з кешу, разом ті самі 59.

У бекенді та сама ідея називається **кешуванням**: результат повільного запиту до бази чи API зберігають і віддають знову. В уроці 31 — через Redis.

### Приклад Р.4. Таблиця знизу вгору — і межа рекурсії

Замість спуску від `n` заповнюємо список циклом: `table[0] = 1`, `table[1] = 1`, кожна наступна комірка — сума двох попередніх. Таблиці для `n` потрібна `n + 1` комірка.

In [ ]:
def stairs_table(n):
    table = [0] * (n + 1)
    table[0] = 1
    if n >= 1:
        table[1] = 1
    for i in range(2, n + 1):
        table[i] = table[i - 1] + table[i - 2]
    return table[n]

In [ ]:
import sys

print("таблиця, n = 30:", stairs_table(30))
print("межа рекурсії:", sys.getrecursionlimit())
stairs_cached.cache_clear()
try:
    stairs_cached(3000)
except RecursionError as error:
    print("кеш, n = 3000:", type(error).__name__, error)
print("таблиця, n = 3000:", len(str(stairs_table(3000))), "цифр у відповіді")

Кеш прибирає **повторну роботу**, але не **глибину**: 3000 вкладених викликів перевищують межу (і 1000, і 3000 у Jupyter). Таблиця без рекурсії рахує 3000 сходинок миттєво.

**Два питання перед будь-якою таблицею:**

1. Що означає одна комірка? — «`table[i]` — скільки способів піднятися на `i` сходинок».
2. Як порахувати комірку з попередніх? — «`table[i] = table[i - 1] + table[i - 2]`».

Розбити задачу на менші, кожну розв'язати раз і перевикористати відповіді — це **динамічне програмування**. Далі — те саме на задачі водія.

---
## 0. Дані й модель

| Замовлення | Час | Заробіток |
|---|---|---:|
| А | 09:00–10:00 | 200 грн |
| Б | 10:00–11:00 | 200 грн |
| В | 09:00–11:00 | 550 грн |
| Г | 11:00–12:00 | 200 грн |

Спрощення: інтервали фіксовані; одночасно — одне замовлення; кінець о 10:00 дозволяє почати о 10:00; дорогу й витрати не враховуємо; максимізуємо винагороду. Час — цілі години.

Замовлення — `NamedTuple` (урок 19): незмінний кортеж з іменованими полями. `names` і `total` — дві маленькі допоміжні функції для друку розкладів.

In [ ]:
from typing import NamedTuple


class Order(NamedTuple):
    name: str
    start: int   # година початку
    end: int     # година завершення
    reward: int  # грн


orders = [
    Order("А", 9, 10, 200),
    Order("Б", 10, 11, 200),
    Order("В", 9, 11, 550),
    Order("Г", 11, 12, 200),
]


def names(schedule):
    return [o.name for o in schedule]


def total(schedule):
    return sum(o.reward for o in schedule)

In [ ]:
print(names(orders), total(orders))
print(orders[2], orders[2].end - orders[2].start, "год")

Усі чотири замовлення разом «коштують» 1150 грн, але виконати всі не можна — частина перетинається. В триває 2 години.

### Приклад 0.1. Часова схема текстом

Щоб бачити перетини, намалюємо кожне замовлення смужкою: одна клітинка — одна година.

In [ ]:
def timeline(orders, first=None, last=None):
    """Текстова часова схема: █ — година зайнята замовленням."""
    first = min(o.start for o in orders) if first is None else first
    last = max(o.end for o in orders) if last is None else last
    hours = range(first, last)
    print("     " + "".join(f"{h:<3}" for h in hours))
    for o in orders:
        bar = "".join("██ " if o.start <= h < o.end else "·· " for h in hours)
        print(f"{o.name:>3}  {bar} {o.reward} грн")

In [ ]:
timeline(orders)

А і Б стоять «встик» (А закінчується о 10, Б о 10 починається), В накриває обидві, Г — окремо після 11.

**🔮 Прогноз:** яку найбільшу **кількість** замовлень може виконати водій і скільки найбільше **заробити**? Запиши обидва розклади.

<details>
<summary>Відповідь</summary>

3 замовлення (А → Б → Г, 600 грн) і 750 грн (В → Г). Різні розклади для різних цілей — про це весь урок.

</details>

---
## 1. Сумісність

### Приклад 1.1. `compatible(a, b)`

Два замовлення сумісні, якщо одне закінчується **не пізніше**, ніж починається інше — у будь-якому порядку. Знак `<=`, а не `<`, — це правило моделі «кінець о 10:00 дозволяє почати о 10:00».

In [ ]:
def compatible(a, b):
    """Два замовлення не перетинаються: одне закінчується до початку іншого."""
    return a.end <= b.start or b.end <= a.start

In [ ]:
А, Б, В, Г = orders
print(compatible(А, Б), compatible(Б, А))      # дотичні межі — можна
print(compatible(А, В), compatible(В, Б))      # перетин
print(compatible(В, Г), compatible(В, В))
assert compatible(А, Б) and compatible(Б, А)
assert not compatible(А, В) and not compatible(В, Б)
assert compatible(В, Г) and not compatible(В, В)

`True True` — порядок аргументів не важливий; `False False` — А і В обидва тривають о 9-й, В і Б — о 10-й; В і Г сумісні, а замовлення саме з собою — ні (інтервал перетинає сам себе).

### Приклад 1.2. Таблиця сумісності

Для малої зміни зручно побачити всі пари одразу: `+` — можна поєднати, `·` — перетинаються.

In [ ]:
def compatibility_table(orders):
    print("    " + "  ".join(o.name for o in orders))
    for a in orders:
        row = "  ".join("+" if compatible(a, b) else "·" for b in orders)
        print(f"{a.name:>2}  {row}")

In [ ]:
compatibility_table(orders)

Таблиця симетрична відносно діагоналі (сумісність не залежить від порядку), на діагоналі — `·`. В сумісне лише з Г, а Г — з усіма: тому Г буде в обох найкращих розкладах.

---
## 2. Мета 1: найбільше замовлень (greedy)

Правило: відсортуй за **часом завершення**, бери замовлення, якщо воно починається не раніше, ніж закінчилось останнє взяте. Ніколи не повертайся до прийнятих рішень.

**🔮 Прогноз:** у якому порядку `sorted(orders, key=lambda o: o.end)` поставить Б і В (обидва закінчуються об 11)? Які замовлення візьме жадібний алгоритм?

<details>
<summary>Відповідь</summary>

Б, потім В: `sorted` **стабільний** і зберігає початковий порядок рівних. Жадібний візьме А (кінець 10), Б (10 ≥ 10), пропустить В (9 < 11), візьме Г (11 ≥ 11).

</details>

### Приклад 2.1. `max_orders(orders)`

`last_end` — коли водій звільниться; `None` — ще нічого не взято. Одне сортування й один прохід: `O(n log n)`.

In [ ]:
def max_orders(orders):
    chosen = []
    last_end = None
    for order in sorted(orders, key=lambda o: o.end):
        if last_end is None or order.start >= last_end:
            chosen.append(order)
            last_end = order.end
    return chosen

In [ ]:
print(names(sorted(orders, key=lambda o: o.end)))
print(names(max_orders(orders)), total(max_orders(orders)), "грн")
print(max_orders([]))

Порядок після сортування — `['А', 'Б', 'В', 'Г']` (Б раніше за В, як і в початковому списку). Розклад — А, Б, Г: три замовлення, 600 грн. Порожній вхід дає порожній розклад — без особливих випадків у коді.

### Приклад 2.2. Покроковий журнал рішень

Та сама логіка, але кожне рішення друкується: що перевіряли і чому взяли чи пропустили.

In [ ]:
def max_orders_trace(orders):
    chosen, last_end = [], None
    for order in sorted(orders, key=lambda o: o.end):
        free = last_end is None or order.start >= last_end
        check = "вільно" if last_end is None else f"{order.start} >= {last_end}?"
        print(f"{order.name} {order.start}–{order.end}: {check:12} → {'беремо' if free else 'пропускаємо'}")
        if free:
            chosen.append(order)
            last_end = order.end
    return chosen

In [ ]:
result = max_orders_trace(orders)
print("розклад:", names(result))

Чотири кроки — чотири рішення, жодне не переглядається. В пропущено, бо починається о 9, а водій зайнятий до 11. Чому це правило дає максимум — **аргумент обміну** в книзі; у розділі 8 ми ще й перевіримо його перебором.

### Приклад 2.3. Інші «очевидні» правила ламаються

`greedy_by(orders, key)` — загальний жадібний шаблон: переглядаємо замовлення в порядку `key` і беремо все, що сумісне з уже взятим. Підставимо інші правила.

In [ ]:
def greedy_by(orders, key):
    """Жадібно: переглядаємо за key і беремо все, що сумісне з уже взятим."""
    chosen = []
    for order in sorted(orders, key=key):
        if all(compatible(order, c) for c in chosen):
            chosen.append(order)
    return chosen


early = [Order("Л", 8, 12, 0), Order("М", 9, 10, 0), Order("Н", 10, 11, 0)]
short = [Order("П", 8, 11, 0), Order("Р", 10, 12, 0), Order("С", 11, 14, 0)]

In [ ]:
print("найраніший початок:", names(greedy_by(early, key=lambda o: o.start)))
print("найраніший кінець: ", names(max_orders(early)))
print()
print("найкоротше:        ", names(greedy_by(short, key=lambda o: o.end - o.start)))
print("найраніший кінець: ", names(max_orders(short)))
timeline(short)

- «Найраніший початок» бере Л (8–12), і воно блокує обидва короткі замовлення: 1 проти 2.
- «Найкоротше» бере Р (10–12, лише 2 години), яке перетинається і з П, і з С: знову 1 проти 2. Схема показує, що П і С стоять встик об 11.

**Правило:** жадібні правила легко вигадати й так само легко помилитися — їх доводять або хоча б перевіряють перебором.

---
## 3. Зміна мети: гроші

### Приклад 3.1. Скільки заробляє розклад «за кількістю»

In [ ]:
most = max_orders(orders)
print("жадібний за кількістю:", names(most), total(most), "грн")
print("В + Г:                ", total([В, Г]), "грн")

Три замовлення дають 600 грн, а **два** — 750. Жадібний алгоритм не помилився: він розв'язував **іншу** задачу. Правило «найраніший кінець» оптимальне, лише коли всі замовлення важать однаково.

### Приклад 3.2. «Просто бери найдорожче»?

Нове жадібне правило: переглядати від найдорожчого (`key=-reward`) і брати все сумісне.

In [ ]:
def most_expensive_first(orders):
    return greedy_by(orders, key=lambda o: -o.reward)


evening = [Order("Д", 9, 13, 500), Order("Е", 9, 11, 300), Order("Ж", 11, 13, 300)]

In [ ]:
print("ранок:", names(most_expensive_first(orders)), total(most_expensive_first(orders)), "грн")
print("вечір:", names(most_expensive_first(evening)), total(most_expensive_first(evening)), "грн")
print("Е + Ж:", total([evening[1], evening[2]]), "грн")
timeline(evening)

Уранці вийшло 750 — правильно, але це збіг. Увечері правило бере Д (500 грн) — і воно забирає час у двох дешевших, які разом дають 600.

### Приклад 3.3. «Найвигідніше за годину»?

Ще спроба: брати спершу замовлення з найбільшим заробітком **за годину**. Здається розумним — враховує і гроші, і час.

In [ ]:
def best_per_hour_first(orders):
    return greedy_by(orders, key=lambda o: -o.reward / (o.end - o.start))


trap = [Order("Т", 9, 10, 300), Order("У", 9, 13, 1000)]

In [ ]:
for o in trap:
    print(o.name, o.reward / (o.end - o.start), "грн/год")
print("за годину:", names(best_per_hour_first(trap)), total(best_per_hour_first(trap)), "грн")
print("лише У:   ", total(trap[1:]), "грн")

Т — 300 грн/год, У — 250 грн/год, тож правило бере Т і більше нічого не може (У перетинається з Т). 300 грн замість 1000. Жадібний алгоритм робить найкращий крок **зараз** і не бачить його наслідків: для заробітку треба порівнювати **цілі варіанти** — що буде, якщо взяти, і що — якщо пропустити.

### Задача 1. Свій контрприклад

Придумай **свою** зміну `my_shift` з 3–5 замовлень, де жадібний за кількістю розклад (`max_orders`) заробляє **менше**, ніж найкращий можливий. Найкращий заробіток порахуй вручну й запиши в `my_best`.

Перевірка в наступній клітинці переконається, що `my_best` справді найкращий, — переглянувши всі підмножини (так працює повний перебір з розділу 8).

In [ ]:
# YOUR CODE HERE


In [ ]:
from itertools import combinations

greedy_money = total(max_orders(my_shift))
true_best = max(total(c) for size in range(len(my_shift) + 1) for c in combinations(my_shift, size)
                if all(compatible(a, b) for a, b in combinations(c, 2)))
print("greedy:", names(max_orders(my_shift)), greedy_money, "грн; твоє найкраще:", my_best, "грн")
assert 3 <= len(my_shift) <= 5, "потрібно 3–5 замовлень"
assert greedy_money < my_best, "на цій зміні жадібний розклад не програє"
assert my_best == true_best, f"my_best — не найкращий заробіток: перебір знайшов {true_best}"
print("✅ Задача 1 розв'язана")

---
## 4. ДП: позначення і `p(i)`

Відсортуй замовлення за `end` і пронумеруй з **одиниці**: замовлення `i` — це `orders[i - 1]`.

- `dp[i]` — найбільший заробіток, якщо дозволено брати лише перші `i` замовлень; `dp[0] = 0` («жодного замовлення» — **база**);
- `p(i)` — скільки перших замовлень закінчуються не пізніше за **початок** замовлення `i`. Усі вони сумісні з `i`, бо список відсортовано за `end`.

Перехід: `dp[i] = max(dp[i - 1], reward(i) + dp[p(i)])` — **пропустити** або **взяти**.

**🔮 Прогноз:** чому дорівнюють `p(1)`, `p(2)`, `p(3)`, `p(4)` для порядку А, Б, В, Г?

<details>
<summary>Відповідь</summary>

`[0, 1, 0, 3]`: до 9 не закінчилось нічого; до 10 — А; до 9 — нічого; до 11 — А, Б, В.

</details>

### Приклад 4.1. `prev_compatible(orders)`

Замовлення **вже відсортовані** за `end`. Для кожного `i` йдемо назад від `i - 1` до `1` і зупиняємось на першому `j`, що закінчився до початку `i`. Оскільки кінці відсортовані, усі `1 … j` теж сумісні. Список має довжину `n + 1`; `p[0]` не використовується.

In [ ]:
def prev_compatible(orders):
    """p[i] для замовлень, уже відсортованих за end. p[0] не використовується."""
    p = [0] * (len(orders) + 1)
    for i in range(1, len(orders) + 1):
        start = orders[i - 1].start
        for j in range(i - 1, 0, -1):
            if orders[j - 1].end <= start:
                p[i] = j
                break
    return p

In [ ]:
by_end = sorted(orders, key=lambda o: o.end)
print(names(by_end), prev_compatible(by_end))
print(prev_compatible([]))
assert prev_compatible(by_end) == [0, 0, 1, 0, 3]

`[0, 0, 1, 0, 3]`: перший нуль — невикористаний `p[0]`, далі `p(1) … p(4)` збігаються з прогнозом. Порожній вхід — `[0]`.

### Приклад 4.2. Звідки кожне `p(i)`

Розпишемо пошук: для кожного замовлення — які попередні вже закінчились до його початку.

In [ ]:
def explain_p(orders):
    orders = sorted(orders, key=lambda o: o.end)
    p = prev_compatible(orders)
    for i, o in enumerate(orders, start=1):
        before = [f"{orders[j - 1].name}(кінець {orders[j - 1].end})" for j in range(1, p[i] + 1)]
        print(f"i={i} {o.name} старт {o.start:>2}: p={p[i]}  закінчились до старту: {', '.join(before) or '—'}")

In [ ]:
explain_p(orders)
print()
explain_p(evening)

Для ранку Г (старт 11) «бачить» усі три попередні, тож `p(4) = 3`. Увечері після сортування порядок Е, Д, Ж (Д і Ж обидва до 13, Д стоїть раніше у вхідному списку): для Ж (старт 11) сумісне лише Е — `p = 1`.

---
## 5. Таблиця ДП

### Приклад 5.1. `max_earnings(orders)`

Сортування → `p` → список `dp` → цикл від `1` до `n`. Порядок вхідних замовлень може бути будь-який: функція сама сортує копію.

In [ ]:
def max_earnings(orders):
    orders = sorted(orders, key=lambda o: o.end)
    p = prev_compatible(orders)
    dp = [0] * (len(orders) + 1)           # dp[0] = 0 — база
    for i in range(1, len(orders) + 1):    # порядок: від менших наборів до більших
        skip = dp[i - 1]
        take = orders[i - 1].reward + dp[p[i]]
        dp[i] = max(skip, take)
    return dp[-1]

In [ ]:
print(max_earnings(orders), max_earnings(list(reversed(orders))), max_earnings([]))
print(max_earnings(evening), max_earnings(trap))

750 для ранку (і для перевернутого списку), 0 для порожнього, 600 увечері й 1000 для пастки з прикладу 3.3 — там, де жадібні правила давали 500 і 300.

**🔮 Прогноз:** заповни таблицю для ранку: `dp[0]` … `dp[4]`. Для кожного рядка запиши «пропустити» і «взяти».

<details>
<summary>Відповідь</summary>

| i | замовлення | p | пропустити | взяти | dp |
|---|---|---|---|---|---|
| 0 | — | — | — | — | 0 |
| 1 | А 200 | 0 | 0 | 200 | 200 |
| 2 | Б 200 | 1 | 200 | 400 | 400 |
| 3 | В 550 | 0 | 400 | 550 | 550 |
| 4 | Г 200 | 3 | 550 | 750 | 750 |

</details>

### Приклад 5.2. Таблиця покроково

`dp_table` рахує те саме, що `max_earnings`, але друкує кожен рядок таблиці: `p(i)`, обидва варіанти й переможця. Перевір свій прогноз.

In [ ]:
def dp_table(orders):
    orders = sorted(orders, key=lambda o: o.end)
    p = prev_compatible(orders)
    dp = [0] * (len(orders) + 1)
    print(" i  замовл.     p   пропустити          взяти   dp    рішення")
    print(" 0  —           —            —              —    0")
    for i in range(1, len(orders) + 1):
        o = orders[i - 1]
        skip, take = dp[i - 1], o.reward + dp[p[i]]
        dp[i] = max(skip, take)
        label = f"{o.name} {o.start}–{o.end}"
        print(f"{i:>2}  {label:<9} {p[i]:>3} {skip:>12}   {o.reward:>4} + {dp[p[i]]:>4}   {dp[i]:>4}    "
              f"{'взяти' if take > skip else 'пропустити'}")
    return dp

In [ ]:
dp = dp_table(orders)
print("dp =", dp)

У кожному рядку перемагає «взяти», але `dp` зростає нерівно: у рядку 3 взяти В (550 + 0) вигідніше, ніж зберегти А + Б (400). Саме тут ДП «передумує» щодо рішень greedy — не змінюючи попередніх рядків, а порівнюючи варіанти.

### Приклад 5.3. Уся зміна: коли перемагає «пропустити»

До ранку додались пообідні замовлення — десять на всю зміну. На більших даних з'являються рядки, де брати **невигідно**.

In [ ]:
shift = [
    Order("А", 9, 10, 200), Order("Б", 10, 11, 200), Order("В", 9, 11, 550),
    Order("Г", 11, 12, 200), Order("Ґ", 12, 14, 380), Order("Д", 11, 13, 300),
    Order("Е", 13, 15, 250), Order("Є", 14, 16, 420), Order("Ж", 15, 17, 260),
    Order("З", 16, 18, 330),
]

In [ ]:
timeline(shift)
print()
dp_shift = dp_table(shift)

Тепер є рядки з рішенням «пропустити»: у них `dp[i]` просто копіює `dp[i - 1]` — найкращий заробіток не змінився від появи нового замовлення. Рядок 7: Е (13–15) дає 250 + 850 = 1100 — менше, ніж 1130 без нього (уже є Г + Ґ). Рядок 9: Ж (15–17) дає 260 + 1130 = 1390 проти 1550 з Є. Відповідь — останній рядок, 1880 грн.

### Приклад 5.4. Помилка на межі: `<` замість `<=`

Колега переписав сумісність «строго». Порахуймо таблицю з таким `p`:

In [ ]:
def prev_compatible_bug(orders):
    p = [0] * (len(orders) + 1)
    for i in range(1, len(orders) + 1):
        for j in range(i - 1, 0, -1):
            if orders[j - 1].end < orders[i - 1].start:      # < замість <=
                p[i] = j
                break
    return p

In [ ]:
p_bug = prev_compatible_bug(by_end)
dp_bug = [0] * 5
for i in range(1, 5):
    dp_bug[i] = max(dp_bug[i - 1], by_end[i - 1].reward + dp_bug[p_bug[i]])
print("p правильне:", prev_compatible(by_end)[1:], " p з помилкою:", p_bug[1:])
print("dp з помилкою:", dp_bug, "→", dp_bug[4], "замість 750")

`p(2)` і `p(4)` зменшились: тепер Б не можна поставити після А, а Г — після Б чи В (дотичні межі вважаються перетином). Найкращим стає В саме — 550. Така помилка проходить на даних «з проміжками», тому в перевірці (розділ 8) обов'язково є набір **«дотичні межі»**.

---
## 6. Який розклад дав максимум

### Приклад 6.1. `best_schedule(orders)`

Число — ще не відповідь водієві: потрібен сам розклад. Заповнюємо `dp`, як у прикладі 5.1, а потім ідемо **з кінця**: якщо `reward(i) + dp[p(i)] > dp[i - 1]` — «взяти» перемогло, замовлення `i` в розкладі, стрибаємо в `p(i)`; інакше — у `i - 1`. Наприкінці розвертаємо список, щоб отримати порядок часу.

In [ ]:
def best_schedule(orders):
    orders = sorted(orders, key=lambda o: o.end)
    p = prev_compatible(orders)
    dp = [0] * (len(orders) + 1)
    for i in range(1, len(orders) + 1):
        dp[i] = max(dp[i - 1], orders[i - 1].reward + dp[p[i]])

    chosen = []
    i = len(orders)
    while i > 0:
        if orders[i - 1].reward + dp[p[i]] > dp[i - 1]:   # «взяти» перемогло
            chosen.append(orders[i - 1])
            i = p[i]
        else:                                             # «пропустити»
            i -= 1
    return list(reversed(chosen))

In [ ]:
for label, data in (("ранок", orders), ("вечір", evening), ("пастка", trap), ("порожньо", [])):
    plan = best_schedule(data)
    print(f"{label:8} {names(plan)}  {total(plan)} грн")

Ранок — В → Г (750), вечір — Е → Ж (600), пастка — лише У (1000), порожньо — `[]`. Сума кожного розкладу збігається з `max_earnings`.

### Приклад 6.2. Шлях назад по таблиці

Надрукуємо кожен крок відновлення на всій зміні: де стоїмо, що порівнюємо і куди стрибаємо.

In [ ]:
def trace_back(orders):
    orders = sorted(orders, key=lambda o: o.end)
    p = prev_compatible(orders)
    dp = [0] * (len(orders) + 1)
    for i in range(1, len(orders) + 1):
        dp[i] = max(dp[i - 1], orders[i - 1].reward + dp[p[i]])
    i = len(orders)
    while i > 0:
        o = orders[i - 1]
        take = o.reward + dp[p[i]]
        if take > dp[i - 1]:
            print(f"i={i:>2} {o.name}: взяти {take} > пропустити {dp[i - 1]} → беремо, стрибок у p={p[i]}")
            i = p[i]
        else:
            print(f"i={i:>2} {o.name}: взяти {take} <= пропустити {dp[i - 1]} → пропускаємо, i={i - 1}")
            i -= 1
    print("i = 0 — стоп")

In [ ]:
trace_back(shift)
print(names(best_schedule(shift)), total(best_schedule(shift)), "грн")

Стрибки в `p(i)` перескакують цілі шматки таблиці: після кожного взятого замовлення ми одразу опиняємось серед тих, що закінчились до його початку. Розклад — В, Г, Ґ, Є, З: 5 замовлень, 1880 грн.

### Задача 2. Перерва між замовленнями

Водієві потрібна **перерва** `rest` годин між замовленнями: після замовлення, що закінчилось о 10:00, при `rest=1` наступне можна почати не раніше 11:00.

Напиши `max_earnings_rest(orders, rest=0)` — те саме ДП, що в прикладі 5.1, але замовлення `j` сумісне з `i`, якщо `orders[j - 1].end + rest <= start`. Змінюється лише умова пошуку `p(i)`; решта алгоритму — та сама. Можна написати допоміжну функцію або рахувати `p` прямо всередині.

Перевірка — у наступній клітинці.

In [ ]:
def max_earnings_rest(orders, rest=0):
    # YOUR CODE HERE
    pass


In [ ]:
print(max_earnings_rest(orders), max_earnings_rest(orders, rest=1), max_earnings_rest(shift, rest=1))
assert max_earnings_rest(orders) == 750, "при rest=0 результат має збігатися з max_earnings"
assert max_earnings_rest(shift, rest=0) == max_earnings(shift) == 1880
assert max_earnings_rest(orders, rest=1) == 550, "після А (до 10) при rest=1 можна взяти лише Г; В саме — 550"
assert max_earnings_rest([], rest=2) == 0
assert max_earnings_rest(evening, rest=1) == 500, "увечері при rest=1 Е і Ж стоять встик — лишається Д"
print("✅ Задача 2 розв'язана")

---
## 7. Те саме зверху вниз: рекурсія з пам'яттю

### Приклад 7.1. Формула рекурсією — і скільки це коштує

Формулу `best(i) = max(best(i - 1), reward(i) + best(p(i)))` можна записати рекурсією майже дослівно. Порахуймо виклики на 20 замовленнях «ланцюжком»: кожне перетинається з сусідами, тож `p(i) = i - 2`.

In [ ]:
calls = 0


def best_plain(i, orders, p):
    global calls
    calls += 1
    if i == 0:
        return 0
    return max(best_plain(i - 1, orders, p),
               orders[i - 1].reward + best_plain(p[i], orders, p))


chain = [Order(f"#{k}", k, k + 2, 100) for k in range(1, 21)]

In [ ]:
chain_sorted = sorted(chain, key=lambda o: o.end)
chain_p = prev_compatible(chain_sorted)
print(chain_p[:8], "...")
for n in (10, 15, 20):
    calls = 0
    result = best_plain(n, chain_sorted, chain_p)
    print(f"n = {n}: результат {result}, викликів {calls}")

`p` — `[0, 0, 0, 1, 2, 3, …]`, тобто `p(i) = i - 2`. Рекурсія рахує `best(i - 1)` і `best(i - 2)` — точнісінько як `fib` з уроку 23: 10 замовлень — 287 викликів, 15 — 3193, 20 — уже 35 421. Кожні +5 замовлень множать роботу приблизно в 11 разів.

### Приклад 7.2. `lru_cache`: кожна підзадача — один раз

`functools.lru_cache` запам'ятовує вже пораховані `best(i)`. Функцію `best` визначаємо **всередині**, щоб кеш жив лише для цього набору замовлень.

In [ ]:
from functools import lru_cache


def max_earnings_memo(orders):
    orders = sorted(orders, key=lambda o: o.end)
    p = prev_compatible(orders)

    @lru_cache(maxsize=None)
    def best(i):
        if i == 0:
            return 0
        return max(best(i - 1), orders[i - 1].reward + best(p[i]))

    result = best(len(orders))
    print(best.cache_info())
    return result

In [ ]:
print(max_earnings_memo(chain))
print(max_earnings_memo(shift))

Для ланцюжка — 21 справжнє обчислення (`misses`: `best(0)` … `best(20)`) і 20 відповідей з кешу (`hits`) замість 35 421 виклику. Для всієї зміни — та сама відповідь 1880, що й у таблиці. Це й є динамічне програмування: **та сама формула**, але кожна підзадача рахується один раз.

| | Мемоізація (зверху вниз) | Табуляція (знизу вгору) |
|---|---|---|
| Напрямок | від `best(n)` до бази | від `dp[0]` до `dp[n]` |
| Код | рекурсія + кеш (`lru_cache`, словник) | цикл, що заповнює список |
| Що рахує | лише потрібні підзадачі | усі підзадачі по черзі |
| Ризик | глибина рекурсії (~1000, урок 23) | треба самому задати порядок |

### Приклад 7.3. Межа рекурсії

Ризик з таблиці можна побачити: 10 000 замовлень одне за одним — рекурсія піде на глибину 10 000.

In [ ]:
import sys

long_day = [Order(str(k), k, k + 1, 10) for k in range(10_000)]
print("межа рекурсії:", sys.getrecursionlimit())
try:
    print(max_earnings_memo(long_day))
except RecursionError as error:
    print("RecursionError:", error)
print("табуляція:", max_earnings(long_day))

Мемоізація впала з `RecursionError`: глибина рекурсії обмежена (`sys.getrecursionlimit()` — зазвичай 1000, у Jupyter буває 3000). Табуляція спокійно дала 100 000: 10 000 замовлень по 10 грн, усі сумісні. Для довгих наборів цикл надійніший.

**Чотири питання ДП** для нашої задачі:

| Питання | Відповідь |
|---|---|
| **Стан** — що описує підзадачу? | `i`: дозволено брати лише перші `i` замовлень |
| **Перехід** — як відповідь складається з менших? | `dp[i] = max(dp[i-1], reward(i) + dp[p(i)])` |
| **База** — що відомо без обчислень? | `dp[0] = 0` |
| **Порядок** — у якій послідовності рахувати? | `i` від 1 до `n`: `p(i) < i`, тож усе потрібне вже пораховано |

Стан тут — одна змінна `i`, тож таблиця — рядок. Скільки змінних буває в стані і що тоді з таблицею — розділ 10.6.

---
## 8. Перевірка перебором

Повний перебір усіх `2ⁿ` підмножин — повільний, але очевидно правильний **суддя** для малих `n` (урок 26: швидке рішення порівнюють з повільним).

### Приклад 8.1. `brute_force(orders)` і граничні набори

`combinations(orders, size)` дає всі підмножини розміру `size`; підмножина допустима, якщо **усі пари** в ній сумісні.

In [ ]:
from itertools import combinations


def brute_force(orders):
    """Найбільший заробіток серед усіх сумісних підмножин. O(2ⁿ · n²)."""
    best = 0
    for size in range(len(orders) + 1):
        for combo in combinations(orders, size):
            if all(compatible(a, b) for a, b in combinations(combo, 2)):
                best = max(best, total(combo))
    return best


cases = {
    "порожній список": [],
    "усі сумісні": [Order("x", 9, 10, 100), Order("y", 10, 11, 150), Order("z", 11, 12, 50)],
    "усі перетинаються": [Order("x", 9, 12, 100), Order("y", 10, 13, 150), Order("z", 11, 14, 50)],
    "дотичні межі": orders,
    "однаковий кінець": [Order("x", 9, 11, 100), Order("y", 10, 11, 150)],
    "пастка за годину": trap,
}

In [ ]:
for name, data in cases.items():
    print(f"{name:18} ДП = {max_earnings(data):4}  перебір = {brute_force(data):4}")
    assert max_earnings(data) == brute_force(data)

На всіх граничних наборах відповіді однакові: порожній — 0, усі сумісні — сума всіх (300), усі перетинаються — найдорожче одне (150), однаковий кінець — дорожче з двох.

### Приклад 8.2. Сотні випадкових змін

Граничні випадки — ті, що ми придумали. Випадкові дані знаходять ті, про які ми не подумали. `random.seed` робить «випадковість» відтворюваною.

In [ ]:
import random


def random_shift(rng, max_orders_count=8):
    data = []
    for k in range(rng.randint(0, max_orders_count)):
        start = rng.randint(8, 20)
        data.append(Order(str(k), start, start + rng.randint(1, 4), rng.randint(1, 10) * 50))
    return data

In [ ]:
rng = random.Random(26)
mismatches = 0
for trial in range(300):
    data = random_shift(rng)
    if max_earnings(data) != brute_force(data) or total(best_schedule(data)) != max_earnings(data):
        mismatches += 1
print("перевірено 300 змін, розбіжностей:", mismatches)

0 розбіжностей: і число `max_earnings`, і сума розкладу `best_schedule` збігаються з перебором. Якби ДП мало помилку (як `<` у прикладі 5.4), такий цикл майже напевно знайшов би зміну, де відповіді розходяться. У файлі з тестами це стане `@pytest.mark.parametrize` по наборах з уроку 26.

### Приклад 8.3. Перебір підтверджує і greedy

Той самий суддя перевіряє й жадібний алгоритм — але для **кількості**: найбільший розмір сумісної підмножини.

In [ ]:
def brute_force_count(orders):
    """Найбільша кількість попарно сумісних замовлень."""
    for size in range(len(orders), -1, -1):
        for combo in combinations(orders, size):
            if all(compatible(a, b) for a, b in combinations(combo, 2)):
                return size
    return 0

In [ ]:
rng = random.Random(27)
greedy_ok = sum(len(max_orders(d)) == brute_force_count(d) for d in (random_shift(rng) for _ in range(300)))
print("greedy за кількістю збігся з перебором:", greedy_ok, "з 300")

rng = random.Random(27)
money_ok = sum(total(max_orders(d)) == brute_force(d) for d in (random_shift(rng) for _ in range(300)))
print("greedy за кількістю дав найбільший заробіток:", money_ok, "з 300")

За **кількістю** greedy збігся з перебором на всіх 300 змінах — так і обіцяє аргумент обміну. А за **грошима** той самий розклад виявився найкращим лише в частині змін: на решті він заробляє менше. Одна перевірка показує і силу, і межу жадібного правила.

### Приклад 8.4. Уся зміна: кількість проти грошей

Зводимо разом три способи на зміні з десяти замовлень. Перебір тут — `2¹⁰ = 1024` підмножини, ще швидко.

In [ ]:
most = max_orders(shift)
best = best_schedule(shift)
print("найбільше замовлень: ", names(most), len(most), "шт.,", total(most), "грн")
print("найбільший заробіток:", names(best), len(best), "шт.,", total(best), "грн")
print("ДП:", max_earnings(shift), " перебір:", brute_force(shift))

- **greedy** дає 6 замовлень — більше не можна, але грошей 1730;
- **ДП** жертвує одним замовленням (А + Б → В) і заробляє на 150 грн більше;
- **перебір** підтверджує 1880.

Вибір між розкладами — бізнес-рішення: рейтинг чи гроші. Алгоритм лише чесно відповідає на поставлене питання.

---
## 9. Складність і `bisect`

### Приклад 9.1. Звідки `O(n²)`

Порахуймо кроки внутрішнього циклу `prev_compatible` на найгіршому наборі: усі замовлення починаються одночасно й перетинаються, тож пошук щоразу доходить до початку списку.

In [ ]:
def p_steps(orders):
    steps = 0
    for i in range(1, len(orders) + 1):
        for j in range(i - 1, 0, -1):
            steps += 1
            if orders[j - 1].end <= orders[i - 1].start:
                break
    return steps

In [ ]:
for n in (500, 1000, 2000):
    worst = [Order(str(k), 0, k + 1, 1) for k in range(n)]
    print(f"n = {n:>4}: кроків {p_steps(worst):>9}")

Удвічі більше замовлень — учетверо більше кроків (дослід подвоєння з уроку 9): 124 750 → 499 500 → 1 999 000. Це `n·(n−1)/2`, тобто `O(n²)`. Заповнення самої таблиці — лише `O(n)`, тож вузьке місце — пошук `p(i)`.

### Приклад 9.2. `bisect_right` на списку кінців

Але кінці **відсортовані**, а `p(i)` — це кількість кінців `<= start`. Саме це рахує `bisect.bisect_right(a, x)` з уроку 12 — двійковим пошуком, за `O(log n)`. Необов'язкові `lo` і `hi` обмежують пошук частиною списку `a[lo:hi]`.

In [ ]:
import bisect

ends = [10, 11, 11, 12, 13, 13]
for x in (9, 10, 11, 12):
    print(f"кінців <= {x:>2}: {bisect.bisect_right(ends, x)}")
print("серед перших 2 кінців <= 11:", bisect.bisect_right(ends, 11, 0, 2))

`bisect_right` повертає позицію **після** усіх рівних: для 11 — 3 (кінці 10, 11, 11). З `hi=2` пошук іде лише по `ends[0:2]` і дає 2. Межа `hi` знадобиться, щоб не зарахувати саме замовлення `i` чи пізніші з таким самим кінцем.

### Задача 3. `max_earnings_fast(orders)`

Напиши те саме ДП, що в прикладі 5.1, але `p(i)` знаходь через `bisect.bisect_right` по списку кінців `ends` — **без вкладених циклів**. Кількість замовлень серед перших `i - 1`, що закінчились до старту замовлення `i`: `bisect.bisect_right(ends, start, 0, i - 1)`.

Перевірка — у наступній клітинці: відомі відповіді плюс 300 випадкових змін, де результат має збігтися з `max_earnings`.

In [ ]:
def max_earnings_fast(orders):
    # YOUR CODE HERE
    pass


In [ ]:
print(max_earnings_fast(orders), max_earnings_fast(shift))
assert max_earnings_fast(orders) == 750
assert max_earnings_fast(shift) == 1880
assert max_earnings_fast([]) == 0
rng = random.Random(8)
for trial in range(300):
    data = random_shift(rng, max_orders_count=12)
    assert max_earnings_fast(data) == max_earnings(data), data
print("✅ Задача 3 розв'язана")

| Алгоритм | Час | Що дає |
|---|---|---|
| `max_orders` (greedy) | `O(n log n)` — сортування | найбільша **кількість** |
| `brute_force` | `O(2ⁿ · n²)` | будь-яку мету, але лише для малих `n` |
| `max_earnings` з простим `p(i)` | `O(n²)` у гіршому разі | найбільший **заробіток** |
| те саме з `bisect` для `p(i)` | `O(n log n)` | те саме, швидше |

---
## 10. Ще одна задача: решта монетами

Сходи питали, **скільки** способів. Тепер — **який найкращий**: найменше монет для суми.

### Приклад 10.1. Greedy: найбільша монета, що влазить

Касир бере найбільшу монету чи купюру, яка влазить, і повторює. Журнал показує кожен крок.

In [ ]:
HRYVNIA = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]


def greedy_change(amount, coins, log=False):
    used = []
    for coin in sorted(coins, reverse=True):
        while amount >= coin:
            amount -= coin
            used.append(coin)
            if log:
                print(f"  беремо {coin:>4}, лишилось {amount}")
    return used

In [ ]:
print(greedy_change(68, HRYVNIA, log=True))

68 = 50 + 10 + 5 + 2 + 1. Вибір, що здається найкращим зараз, і жодних повернень — як «найраніший кінець» у розділі 2.

### Приклад 10.2. Контрприклад: монети `[1, 3, 4]`

In [ ]:
print("greedy:", greedy_change(6, [1, 3, 4], log=True))
print("а можна:", [3, 3])

Greedy дав три монети, а вистачає двох. Один такий вхід — **контрприклад** — доводить, що для цих монет greedy не працює. Чесний спосіб знати напевно — порівняти з методом, який завжди правий, сума за сумою. Так само перевіряй будь-яке хитре швидке рішення, і те, що запропонував ШІ-асистент.

### Приклад 10.3. Таблиця `best[a]`

1. **Що означає комірка?** — `best[a]` — найменше монет для суми `a`.
2. **Як її порахувати?** — якщо остання монета `c`, потрібно `best[a - c] + 1`; пробуємо кожну й беремо найменше.

`best[0] = 0`, решта стартує з `float("inf")` — «поки що не можна скласти».

In [ ]:
def min_coins_table(amount, coins, log=False):
    best = [float("inf")] * (amount + 1)
    best[0] = 0
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                best[a] = min(best[a], best[a - c] + 1)
        if log:
            print(f"  best[{a}] = {best[a]}")
    return best

In [ ]:
print(min_coins_table(6, [1, 3, 4], log=True))
print("трійку двійками:", min_coins_table(3, [2]))

Таблиця для `[1, 3, 4]` — `0, 1, 2, 1, 1, 2, 2`: шістці досить двох монет. Якщо остання комірка лишилась `inf`, суму не скласти взагалі.

### Приклад 10.4. Greedy проти таблиці, сума за сумою

In [ ]:
def greedy_failures(coins, limit):
    best = min_coins_table(limit, coins)
    return [a for a in range(1, limit + 1) if len(greedy_change(a, coins)) > best[a]]

In [ ]:
print("[1, 3, 4] до 12:", greedy_failures([1, 3, 4], 12))
print("гривня до 1000:", greedy_failures(HRYVNIA, 1000))

Для гривні greedy збігся з таблицею на кожній сумі до 1000 (такі набори монет називають **канонічними**), для `[1, 3, 4]` ламається вже на 6.

### Приклад 10.5. Який саме набір: запам'ятати вибір

При кожному новому мінімумі запам'ятовуємо монету в `last[a]`, потім ідемо назад від `amount` — як відновлення розкладу в розділі 6.

In [ ]:
def change_coins(amount, coins, log=False):
    best = [float("inf")] * (amount + 1)
    best[0] = 0
    last = [None] * (amount + 1)
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and best[a - c] + 1 < best[a]:
                best[a] = best[a - c] + 1
                last[a] = c
    if best[amount] == float("inf"):
        return None
    used = []
    while amount > 0:
        if log:
            print(f"  сума {amount}: остання монета {last[amount]} → {amount - last[amount]}")
        used.append(last[amount])
        amount -= last[amount]
    return used

In [ ]:
print(change_coins(6, [1, 3, 4], log=True))
print(change_coins(7, [2, 4]))

### Приклад 10.6. Матрична параметризація: скільки змінних у стані

Підготовку ДП розкладають на п'ять кроків:

| Крок | Що зробити | Монети |
|---|---|---|
| 1. Параметризація | скільки змінних описує підзадачу — стільки вимірів у таблиці | сума `a` |
| 2. Розмір простору | підзадач має бути поліноміально, а не `2ⁿ` | `A + 1` комірка |
| 3. Порядок обходу | менші підзадачі готові раніше за більші | `a` від 1 вгору |
| 4. Рекурсивна формула | як більша відповідь складається з менших | `best[a] = min(best[a - c] + 1)` |
| 5. Запам'ятати вибір | для `min`/`max` зберегти, який варіант переміг | `last[a]` |

Потім — заповнити таблицю знизу вгору й пройти назад за збереженим вибором. Кроки 1–4 — ті самі чотири питання з розділу 7: стан, перехід, база, порядок.

Крок 1 визначає форму таблиці:

```mermaid
flowchart TD
    classDef step     fill:#eceff1,stroke:#546e7a,stroke-width:1px,color:#1a1a1a;
    classDef decision fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#1a1a1a;
    classDef success  fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#1a1a1a;
    classDef error    fill:#ffebee,stroke:#c62828,stroke-width:3px,color:#1a1a1a;

    Q["задача"] --> V{"скільки змінних треба,<br>щоб назвати одну підзадачу?"}
    V -- "одна" --> ONE
    V -- "дві" --> TWO

    subgraph ONE["1 вимір → рядок dp[i]"]
        direction TB
        S1["сходи: i — скільки сходинок<br>table[0..n] · n + 1 комірка"]
        S2["водій: i — перші i замовлень<br>dp[0..n] · n + 1 комірка"]
        S3["монети: a — сума<br>best[0..A] · A + 1 комірка"]
    end

    subgraph TWO["2 виміри → матриця dp[i][j]"]
        direction TB
        T1["монети: k — перші k номіналів,<br>a — сума · (m + 1)·(A + 1)"]
        T2["водій з пальним: i — перші i замовлень,<br>f — залишок пального · (n + 1)·(F + 1)"]
    end

    ONE --> P{"комірок поліноміально<br>багато?"}
    TWO --> P
    P -- "так" --> OK["таблиця вміщається:<br>порядок → формула → вибір"]
    P -- "ні, ~2ⁿ" --> NO["стан описано задетально:<br>шукай менше змінних"]

    class Q,S1,S2,S3,T1,T2 step
    class V,P decision
    class OK success
    class NO error
```

Монети можна описати й **двома** змінними: `best[k][a]` — найменше монет для суми `a`, якщо дозволено лише перші `k` номіналів. Рядок `k` або не бере номінал `c = coins[k - 1]` (`best[k - 1][a]`), або бере його ще раз (`best[k][a - c] + 1`). Це твоя Задача 4.

### Задача 4. Матриця `best[k][a]`

Напиши `min_coins_matrix(amount, coins)`, що повертає **матрицю** з `len(coins) + 1` рядків і `amount + 1` стовпців:

- рядок 0 — без монет: `best[0][0] = 0`, решта `float("inf")`;
- у кожному рядку `best[k][0] = 0`;
- `best[k][a] = min(best[k - 1][a], best[k][a - c] + 1)`, де `c = coins[k - 1]`, друга частина — лише якщо `c <= a`.

Перевірка — у наступній клітинці.

In [ ]:
def min_coins_matrix(amount, coins):
    # YOUR CODE HERE
    pass


In [ ]:
matrix = min_coins_matrix(6, [1, 3, 4])
for k, row in enumerate(matrix):
    print(k, row)
inf = float("inf")
assert len(matrix) == 4 and all(len(row) == 7 for row in matrix), "розмір: (m + 1) × (A + 1)"
assert matrix[0] == [0, inf, inf, inf, inf, inf, inf], "рядок 0 — без монет"
assert matrix[2] == [0, 1, 2, 1, 2, 3, 2], "лише [1, 3]: 4 = 3 + 1, 5 = 3 + 1 + 1"
assert matrix[-1] == min_coins_table(6, [1, 3, 4]), "останній рядок = одновимірна таблиця"
assert min_coins_matrix(3, [2])[-1][3] == inf
assert min_coins_matrix(1000, HRYVNIA)[-1] == min_coins_table(1000, HRYVNIA)
print("✅ Задача 4 розв'язана")

Останній рядок матриці збігається з таблицею з прикладу 10.3: монету можна брати скільки завгодно разів, тож другий вимір тут зайвий. Він потрібен, коли кожну монету можна взяти **лише раз** (задача про рюкзак). У стані — стільки змінних, скільки треба для однозначної відповіді, і не більше: кожен вимір множить розмір таблиці.

---
## 11. Як читати помилки теми

| Що бачиш | Що сталося | Що робити |
|---|---|---|
| `RecursionError` навіть з `@cache` | кеш прибирає повторну роботу, а не глибину (приклад Р.4) | таблиця |
| `TypeError: unhashable type: 'list'` | `@cache` отримав список — кеш тримає аргументи як ключі словника | передати кортеж |
| помилки немає, але відповідь 0 для будь-якої суми | таблицю почали нулями замість `float("inf")` | `[float("inf")] * (amount + 1)` |
| `IndexError: list index out of range` (або `list assignment index out of range`) | для `n` потрібна `n + 1` комірка | `[0] * (n + 1)` |

### Приклад 11.1. Список у функції з `@cache`

In [ ]:
@cache
def min_coins_cached(amount, coins):
    if amount == 0:
        return 0
    options = [min_coins_cached(amount - c, coins) + 1 for c in coins if c <= amount]
    return min(options, default=float("inf"))

In [ ]:
try:
    print(min_coins_cached(6, [1, 3, 4]))
except TypeError as error:
    print("список:", type(error).__name__, error)
print("кортеж:", min_coins_cached(6, (1, 3, 4)))

### Приклад 11.2. Таблиця, почата нулями

In [ ]:
def min_coins_zeros(amount, coins):
    best = [0] * (amount + 1)          # помилка: має бути float("inf")
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                best[a] = min(best[a], best[a - c] + 1)
    return best

In [ ]:
print("нулі:", min_coins_zeros(6, [1, 3, 4]))
print("inf: ", min_coins_table(6, [1, 3, 4]))

`min(0, щось + 1)` завжди 0: жоден варіант не менший за стартове значення.

### Приклад 11.3. Таблиця на `n` комірок замість `n + 1`

In [ ]:
def stairs_short(n):
    table = [0] * n                    # помилка: має бути n + 1
    table[0] = 1
    table[1] = 1
    for i in range(2, n + 1):
        table[i] = table[i - 1] + table[i - 2]
    return table[n]

In [ ]:
try:
    stairs_short(10)
except IndexError as error:
    print(type(error).__name__, error)
print(stairs_table(10))

---
## Самоперевірка

1. Чому правило «найраніший кінець» оптимальне для кількості, але не для заробітку?
2. Що означає `dp[i]` і навіщо нумерація з одиниці?
3. Чому для ДП замовлення сортують саме за `end`?
4. Як з таблиці `dp` дістати сам розклад?
5. Навіщо повільний `brute_force`, якщо є швидке ДП?
6. Чому `<` замість `<=` у пошуку `p(i)` дає 550 замість 750?
7. Чим мемоізація відрізняється від табуляції і коли вона падає?
8. Чому `stairs(30)` без кешу робить 2 692 537 викликів, а з кешем — 59?
9. Як довести, що greedy для монет `[1, 3, 4]` неправильний, і як перевірити greedy для гривні?
10. Скільки вимірів і комірок у таблиці `best[k][a]` і навіщо другий вимір?

<details>
<summary>Відповіді</summary>

1. Для кількості кожне замовлення важить +1, і раніше звільнитись завжди не гірше (аргумент обміну). Для заробітку одне дороге замовлення може бути вигідніше за кілька дешевих — правило цього не бачить (600 проти 750).
2. Найбільший заробіток серед перших `i` замовлень. Нумерація з одиниці дає базу `dp[0] = 0` — «жодного замовлення».
3. Тоді всі замовлення, сумісні з `i` і розміщені перед ним, — це префікс довжини `p(i)`, а найкраще для нього вже пораховано: `dp[p(i)]`.
4. Іти з кінця: якщо «взяти» перемогло — замовлення в розкладі, стрибок у `p(i)`; інакше — `i - 1`.
5. Щоб перевірити ДП на малих граничних і випадкових даних: перебір очевидно правильний.
6. Дотичні замовлення стають «несумісними»: Г не ставиться після Б чи В, і найкращим лишається В саме (приклад 5.4).
7. Та сама формула, але рекурсія від `best(n)` з кешем замість циклу від `dp[0]`; на довгих наборах рекурсія впирається в межу глибини (приклад 7.3).
8. Без кешу ті самі `stairs(k)` рахуються знову й знову — перекриття підзадач. З кешем кожне `n` від 0 до 30 рахується раз (31), решта 28 викликів беруть готову відповідь.
9. Одного контрприкладу досить: 6 = 4 + 1 + 1 замість 3 + 3. Для гривні — порівняти з таблицею на кожній сумі (приклад 10.4).
10. Два виміри, `(m + 1)·(A + 1)` комірок. Для необмеженої кількості монет останній рядок збігається з одновимірною таблицею; другий вимір потрібен, коли кожну монету можна взяти лише раз.

</details>

## Далі

- Теорія, покрокові схеми, аргумент обміну, мемоізація проти табуляції: [Урок 27](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_27/).
- Додатковий ноутбук про модуль `re` і жадібні квантифікатори: `re_lesson_27_greedy_regex.ipynb` у цій самій папці.
- **Урок 28** — потоки, multiprocessing, asyncio: вступ.